In [1]:
import pandas as pd

data = pd.read_csv("final_data.csv", encoding='cp949')

In [2]:
selected_columns = [
    "구간평균조회수",
    "토픽_기반_예상조회수",
    "tag_mean_score",
    "upload_interval_score", "mean_interval_score",
    "요일", "업로드_시각",
    "영상길이(초)", "조회수",
    "topic_score",
    "텍스트_개수", "텍스트_비중",
    "얼굴_개수", "얼굴_비중",
    "주요_감정", "감정_angry", "감정_disgust", "감정_fear",
    "감정_happy", "감정_sad", "감정_surprise", "감정_neutral"
]

data = data[selected_columns].copy()

In [3]:
data.isna().sum()

,0
구간평균조회수,0
토픽_기반_예상조회수,3507
tag_mean_score,0
upload_interval_score,0
mean_interval_score,0
요일,0
업로드_시각,0
영상길이(초),0
조회수,0
topic_score,0


In [4]:
data = data.dropna(subset=['토픽_기반_예상조회수'])
data = pd.get_dummies(data, columns=["요일", "주요_감정"], drop_first=True)
data['업로드_시각'] = pd.to_datetime(data['업로드_시각'], errors='coerce').dt.hour


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25873 entries, 0 to 29379
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   구간평균조회수                25873 non-null  float64
 1   토픽_기반_예상조회수            25873 non-null  float64
 2   tag_mean_score         25873 non-null  float64
 3   upload_interval_score  25873 non-null  float64
 4   mean_interval_score    25873 non-null  float64
 5   업로드_시각                 25873 non-null  int32  
 6   영상길이(초)                25873 non-null  int64  
 7   조회수                    25873 non-null  int64  
 8   topic_score            25873 non-null  float64
 9   텍스트_개수                 25873 non-null  int64  
 10  텍스트_비중                 25873 non-null  float64
 11  얼굴_개수                  25873 non-null  int64  
 12  얼굴_비중                  25873 non-null  float64
 13  감정_angry               25873 non-null  float64
 14  감정_disgust             25873 non-null  float64
 15  감정_fear

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X = data.drop(columns=["조회수"])
y = data["조회수"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"✅ RMSE: {rmse:.2f}")
print(f"✅ R²: {r2:.4f}")

✅ RMSE: 63473.94
✅ R²: 0.5641


In [8]:
# ✅ 1. 라이브러리 로딩
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# ✅ 2. 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ✅ 3. XGBoost 회귀 모델 생성 및 학습
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

# ✅ 4. 예측 및 평가
y_pred = xgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")


RMSE: 64255.77
R²: 0.5533


In [9]:
'''from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
import numpy as np

# ✅ 기본 모델
xgb = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)

# ✅ 탐색할 파라미터 공간
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# ✅ Randomized Search
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42
)

random_search.fit(X, y)

# ✅ 결과 출력
print("Best Parameters:", random_search.best_params_)
print(f"Best RMSE: {-random_search.best_score_:.2f}")'''


Fitting 3 folds for each of 30 candidates, totalling 90 fits


KeyboardInterrupt: 

In [ ]:
!pip install -q optuna optuna-integration[xgboost] xgboost

In [10]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform, randint
import numpy as np

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 탐색 공간
param_dist = {
    "n_estimators":   randint(300, 1000),
    "learning_rate":  uniform(0.01, 0.29),
    "max_depth":      randint(3, 10),
    "min_child_weight": uniform(1e-3, 9.999),
    "subsample":      uniform(0.5, 0.5),
    "colsample_bytree": uniform(0.5, 0.5),
    "gamma":          uniform(0, 0.5),
    "reg_alpha":      uniform(0, 1.0),
    "reg_lambda":     uniform(0.5, 2.5),
}

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    eval_metric="rmse",
    random_state=42,
)

rs = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42,
)

rs.fit(X_train, y_train)              # 🚫 early-stopping 인자 전달 X

print("✅ 최적 파라미터:", rs.best_params_)
print(f"✅ CV 평균 RMSE: {-rs.best_score_:.2f}")

# 테스트셋 평가
best_model = rs.best_estimator_
test_rmse = np.sqrt(mean_squared_error(y_test, best_model.predict(X_test)))
print(f"✅ 테스트셋 RMSE: {test_rmse:,.2f}")


Fitting 3 folds for each of 30 candidates, totalling 90 fits
✅ 최적 파라미터: {'colsample_bytree': np.float64(0.5442462510259598), 'gamma': np.float64(0.0979914312095726), 'learning_rate': np.float64(0.023115913784056037), 'max_depth': 7, 'min_child_weight': np.float64(3.887384219605131), 'n_estimators': 637, 'reg_alpha': np.float64(0.5396921323890798), 'reg_lambda': np.float64(1.9668779141596207), 'subsample': np.float64(0.982627653632069)}
✅ CV 평균 RMSE: 68293.71
✅ 테스트셋 RMSE: 61,817.34


In [ ]:
import xgboost as xgb
print("현재 xgboost 버전 ➜", xgb.__version__)

In [ ]:
best_params = {
    "colsample_bytree": 0.5442462510259598,
    "gamma":            0.0979914312095726,
    "learning_rate":    0.023115913784056037,
    "max_depth":        7,
    "min_child_weight": 3.887384219605131,
    "n_estimators":     637,
    "reg_alpha":        0.5396921323890798,
    "reg_lambda":       1.9668779141596207,
    "subsample":        0.982627653632069,

    # 공통 설정(필요 시 수정)
    "objective":   "reg:squarederror",
    "tree_method": "hist",   # Colab CPU 환경
    "eval_metric": "rmse",
    "random_state": 42
}

In [ ]:
#────────────────────────────────────────────
# 1) 라이브러리 설치(최초 1회) & 임포트
#────────────────────────────────────────────

import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

#────────────────────────────────────────────
# 2) 데이터 준비
#────────────────────────────────────────────
# df = ...  # 이미 DataFrame이 메모리에 있다고 가정
target   = "조회수"
features = [c for c in data.columns if c != target]

# bool → int (LightGBM은 0/1로 자동 캐스팅해도 문제 없지만 명시적으로)
data[features] = data[features].astype({col: int for col in data.select_dtypes('bool').columns})

X = data[features]
y = data[target]

#────────────────────────────────────────────
# 3) 학습-테스트 분할
#────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#────────────────────────────────────────────
# 4) LightGBM 모델 학습
#────────────────────────────────────────────
lgb_model = LGBMRegressor(
    objective="rmse",
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=128,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(X_train, y_train)

#────────────────────────────────────────────
# 5) 성능 평가
#────────────────────────────────────────────
pred = lgb_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f"✅ LightGBM 테스트 RMSE: {rmse:,.2f}")


In [ ]:

import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import randint, uniform

target   = "조회수"
X = data.drop(columns=[target])
y = data[target]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

base_lgb = LGBMRegressor(
    objective="rmse",
    random_state=42,
    n_jobs=-1
)


param_dist = {
    "n_estimators":     randint(600, 1400),
    "learning_rate":    uniform(0.03, 0.07),
    "num_leaves":       randint(128, 512),
    "max_depth":        randint(-1, 15),
    "subsample":        uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_lambda":       uniform(0.0, 2.0),
}


rs = RandomizedSearchCV(
    estimator=base_lgb,
    param_distributions=param_dist,
    n_iter=40,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
    random_state=42
)
rs.fit(X_tr, y_tr)


best_params = rs.best_params_
cv_rmse     = -rs.best_score_
print(" Best Params :", best_params)
print(f" CV Mean RMSE: {cv_rmse:,.2f}")

# ────────────────────────────────
# 8) 테스트셋 평가
# ────────────────────────────────
best_model = rs.best_estimator_
test_rmse  = np.sqrt(mean_squared_error(y_te, best_model.predict(X_te)))
print(f" Test RMSE    : {test_rmse:,.2f}")


In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# ① 최적 파라미터 dict ─ np.float64 → float
best_params = {
    "colsample_bytree": 0.9266084230952957,
    "learning_rate":    0.05976091121438713,
    "max_depth":        4,
    "n_estimators":     705,
    "num_leaves":       387,
    "reg_lambda":       0.3697089110510541,
    "subsample":        0.9908753883293675,
    "objective":        "rmse",
    "random_state":     42,
    "n_jobs":           -1
}

# ② 최종 모델 학습 (전체 train 데이터 사용)
final_lgb = LGBMRegressor(**best_params)
final_lgb.fit(X_tr, y_tr)        # 또는 X, y 전체로 재학습

# ③ 테스트셋 성능
pred = final_lgb.predict(X_te)
rmse = np.sqrt(mean_squared_error(y_te, pred))
print(f"✅ 최종 LightGBM 테스트 RMSE: {rmse:,.2f}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1) LightGBM 모델 객체와 입력 컬럼 확인
model = final_lgb          # 또는 best_model
feat_names = X.columns

# 2) 중요도 추출
imp_df = (
    pd.DataFrame({
        "feature": feat_names,
        "importance": model.feature_importances_
    })
    .sort_values(by="importance", ascending=False)
)

# 3) 상위 15개 시각화
top_df = imp_df.head(15).iloc[::-1]        # barh용 뒤집기

plt.figure(figsize=(8, 6))
plt.barh(top_df["feature"], top_df["importance"])
plt.title("LightGBM 변수 중요도 (상위 15)")
plt.xlabel("Importance (split gain)")
plt.tight_layout()
plt.show()

In [ ]:
!apt-get -qq install -y fonts-nanum
!fc-cache -fv
!rm -rf ~/.cache/matplotlib  # 중요: matplotlib 폰트 캐시 삭제

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family'] = 'NanumGothic'